# Donor Churn Prediction
Predicting which donors are at risk of churning (stopping donations).

## 1. Problem Framing
Donor churn occurs when a previously active donor stops giving. Early identification of at-risk donors allows the organization to deploy retention interventions (targeted outreach, personalised thank-you calls, impact reports) before the donor lapses.

**Target variable:** `churned` — binary (1 = no donation in the last 12 months after at least one prior donation, 0 = still active).

**Success metric:** ROC-AUC ≥ 0.80, Precision-Recall AUC ≥ 0.70 (churn is the positive class).

**Stakeholders:** Development / fundraising team.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# --- Simulated donor dataset ---
n = 800
df = pd.DataFrame({
    'donor_id': range(1, n + 1),
    'tenure_years': np.random.exponential(scale=4, size=n).clip(0.1, 20),
    'total_donations': np.random.randint(1, 50, size=n),
    'avg_gift_amount': np.random.lognormal(mean=4.5, sigma=1.0, size=n),
    'days_since_last_gift': np.random.randint(1, 730, size=n),
    'num_campaigns_responded': np.random.randint(0, 20, size=n),
    'email_open_rate': np.random.beta(2, 5, size=n),
    'event_attendance': np.random.randint(0, 10, size=n),
    'gave_major_gift': np.random.randint(0, 2, size=n),
})

# Derive churn label (heuristic for simulation)
churn_prob = (
    0.3
    + 0.3 * (df['days_since_last_gift'] > 365).astype(float)
    - 0.1 * (df['num_campaigns_responded'] > 5).astype(float)
    - 0.1 * df['email_open_rate']
).clip(0.05, 0.95)
df['churned'] = np.random.binomial(1, churn_prob)

print(df.shape)
print(df['churned'].value_counts())
df.head()

In [ ]:
# Missing value check
print("Missing values:\n", df.isnull().sum())

feature_cols = [
    'tenure_years', 'total_donations', 'avg_gift_amount',
    'days_since_last_gift', 'num_campaigns_responded',
    'email_open_rate', 'event_attendance', 'gave_major_gift'
]

X = df[feature_cols].copy()
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    churned_vals = df.loc[df['churned'] == 1, col]
    active_vals = df.loc[df['churned'] == 0, col]
    axes[i].hist(active_vals, bins=20, alpha=0.6, label='Active', color='steelblue')
    axes[i].hist(churned_vals, bins=20, alpha=0.6, label='Churned', color='tomato')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions by Churn Status', fontsize=14)
plt.tight_layout()
plt.show()

print("Churn rate by campaign response quartile:")
print(df.groupby(pd.qcut(df['num_campaigns_responded'], 4, duplicates='drop'))['churned'].mean())

## 4. Modeling
### 4a. Predictive Model — Random Forest Classifier
### 4b. Causal / Explanatory Model — SHAP (TreeExplainer)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# --- 4a: Predictive model ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_proba = rf.predict_proba(X_test)[:, 1]
y_pred = rf.predict(X_test)
print("Random Forest training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model using SHAP ---
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)

    # shap_values for binary classification is a list [class0, class1]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    shap.summary_plot(sv, X_test, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary — Donor Churn (Class 1 = Churned)')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("shap not installed — skipping SHAP explanation. Install with: pip install shap")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay
)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='Random Forest')
axes[0].set_title('ROC Curve — Donor Churn')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Active', 'Churned'], ax=axes[1])
axes[1].set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='steelblue')
plt.title('Random Forest Feature Importances — Donor Churn')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

# Keep features above mean importance
selected_features = importances[importances >= importances.mean()].index.tolist()
print("Selected features:", selected_features)

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

# Retrain on selected features only
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

rf_final = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight='balanced')
rf_final.fit(X_train_sel, y_train)

with open('models/donor_churn_model.pkl', 'wb') as f:
    pickle.dump({'model': rf_final, 'features': selected_features, 'scaler': scaler}, f)

print("Model saved to models/donor_churn_model.pkl")

# Example scoring of new donors
new_donors = X_test_sel.iloc[:5].copy()
new_donors['churn_probability'] = rf_final.predict_proba(new_donors)[:, 1]
new_donors['risk_tier'] = pd.cut(
    new_donors['churn_probability'],
    bins=[0, 0.4, 0.65, 1.0],
    labels=['Low', 'Medium', 'High']
)
print(new_donors[['churn_probability', 'risk_tier']])